last modified date : 2026.05  
제작 : 모두의연구소

# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가

# 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 직접 구현해 보았습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 데이터셋 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이번 실습이 끝나면 다음을 직접 말할 수 있게 됩니다.
- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 ‘얼마나 좋아졌는지’를 **숫자로** 보여주는 방법

## Step 0 : 설치와 준비  
Day 1과 동일하게 Colab에서 진행한다고 가정합니다.

In [ ]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 248.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 178.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()  # RAGAS가 Colab의 비동기 이벤트 루프와 충돌하지 않도록

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 기간 동안 어떤 프로젝트나 계획이 전면적으로 개선되었나요?']


검색된 문서 수: 6
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [ ]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수 — TODO: 여러분이 직접 채워보세요
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """
    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    # TODO 1: 각 쿼리의 결과 리스트를 순회하면서 문서마다 RRF 점수를 누적해 보세요.
    #   힌트:
    #     for docs in results_per_query:
    #         for rank, doc in enumerate(docs):  # rank 는 0부터
    #             key = doc.page_content
    #             scores[key] += 1.0 / (k + rank + 1)
    #             docs_by_key[key] = doc


    # TODO 2: scores 값이 큰 순서로 정렬해서 상위 top_k 개의 Document 를 반환하세요.
    #   힌트:
    #     ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    #     return [docs_by_key[k] for k, _ in ranked[:top_k]]

    return []


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 시절 2004년에 개선된 것은 무엇인가?

RAG-Fusion top-1 문서:
(아직 TODO 가 비어 있어 결과가 없습니다)


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 종합계획'을 수립하여 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용 차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 이동 편의를 증대시키고, 대기 오염 문제를 완화하는 데 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [ ]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [ ]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [ ]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트 — TODO 1
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    # TODO: 사용자 질문이 외부 문서 검색이 필요하면 YES,
    #       일반 상식·계산·정의 등 LLM 자체 지식으로 답할 수 있으면 NO
    #       오직 한 단어(YES 또는 NO)만 출력하도록 한국어 프롬프트를 채우세요.
    ""
)

# (2) 답변 자가 비평 프롬프트 — TODO 2
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    # TODO: [문서] {context} 와 [답변] {answer} 가 주어졌을 때,
    #       답변이 문서 내용으로 충분히 뒷받침되면 SUPPORTED,
    #       아니면 NOT_SUPPORTED — 한 단어만 출력하도록 한국어 프롬프트를 채우세요.
    ""
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

[1] Retrieve 필요? -> HELLO! HOW CAN I ASSIST YOU TODAY?
[3] 시도 1 — 자가 비평: HELLO! HOW CAN I ASSIST YOU TODAY?

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [13]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [14]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [15]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [16]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.625         0.825
answer_relevancy       0.306         0.255
context_precision      0.692         0.850
context_recall         0.750         0.850

Delta (Advanced - Naive):
faithfulness         0.200
answer_relevancy    -0.051
context_precision    0.158
context_recall       0.100
dtype: float64


### 결과 해석 가이드

## RAGAS 지표별 상세 분석

### 1. 🟢 Faithfulness (충실도 / 환각 방지)
* **Score:** $\text{Naive (0.625)} \rightarrow \text{Advanced (0.825)}$ `[Δ +0.200]`
* **해설:** **가장 성공적인 개선점입니다.** 점수가 `0.20`이나 올랐다는 것은 AI가 '지어내서 하는 거짓말(환각 현상)'이 눈에 띄게 줄었다는 뜻입니다. ⑨번 단계에서 추가한 **Self-RAG(자가 검토 루프)**가 제 역할을 톡톡히 해내어, AI가 답변을 내놓기 전에 문서에 있는 내용인지 스스로 꼼꼼하게 검증했음이 수치로 증명되었습니다.

---

### 2. 🔵 Context Precision (검색 정밀도)
* **Score:** $\text{Naive (0.692)} \rightarrow \text{Advanced (0.850)}$ `[Δ +0.158]`
* **해설:** **예상했던 Advanced RAG의 핵심 효과가 그대로 나타났습니다.** 이 점수가 올랐다는 것은 AI가 답변을 작성할 때 참고하는 top-3 문서 리스트 중에서 **'진짜 정답에 직결되는 핵심 문서'가 맨 상위권(1등)에 예쁘게 잘 배치되었다**는 뜻입니다. ⑦번에서 도입한 **Cross-Encoder Reranker**가 거친 임베딩 검색 결과를 아주 깐깐하게 재정렬해 준 덕분입니다.

---

### 3. 🟣 Context Recall (검색 재현율 / 풍부함)
* **Score:** $\text{Naive (0.750)} \rightarrow \text{Advanced (0.850)}$ `[Δ +0.100]`
* **해설:** 이 점수의 상승은 ④번 **Multi-Query**와 ⑤번 **RAG-Fusion**의 성과입니다. 사용자가 질문을 조금 어설프게 하더라도, AI가 질문을 4개로 확장하여 지식 창고를 넓게 뒤진 덕분에 **정답이 들어있는 진짜 문서를 놓치지 않고 긁어오는 확률**이 `10%` 더 높아졌습니다.

---

### 4. 🔴 Answer Relevancy (답변 관련성 / 집중도)
* **Score:** $\text{Naive (0.306)} \rightarrow \text{Advanced (0.255)}$ `[Δ -0.051]`
* **해설:** **유일하게 점수가 떨어진 지표입니다.** 왜 이런 일이 일어났을까요?
  > Advanced RAG 과정에서 **Multi-Query**와 **HyDE(가상 답변)**를 쓰면서 너무 많은 정보와 다양한 맥락의 문서들이 한꺼번에 참조 데이터로 유입되었을 가능성이 큽니다.
  
  이로 인해 LLM이 답변을 생성할 때, 질문의 핵심 과녁을 바로 찌르기보다는 **주변 배경 설명이나 군더더기 문장을 조금 더 길게 덧붙여 대답했을 확률**이 높습니다. (프롬프트의 제약보다 정보량이 많아져 답변이 다소 장황해진 상태입니다.)

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer (예시)**:  
##  Advanced RAG 성능 개선 핵심 요약

### 1. 가장 크게 개선된 지표: `Faithfulness (충실도 / 환각 방지)`
성적표의 변화량($\Delta$)을 비교했을 때, 가장 압도적인 상승 폭을 기록한 지표는 **충실도(Faithfulness)**입니다.

* **Faithfulness (충실도):** `0.625` $\rightarrow$ `0.825` **[Δ +0.200]** ─── *1등 상승*
* **Context Precision (검색 정밀도):** `0.692` $\rightarrow$ `0.850` **[Δ +0.158]** ─── *2등 상승*
* **Context Recall (검색 재현율):** `0.750` $\rightarrow$ `0.850` **[Δ +0.100]**

---

### 2. 핵심 지표와 적용 기법의 연결 고리

#### 🟢 Faithfulness 점수 폭등 ($\Delta$ +0.200) $\rightarrow$ `⑨번 Self-RAG (자가 검토)`
* **연결 원리:** `Faithfulness`는 AI가 답변을 작성할 때 허공에서 말을 지어내는 **'환각(Hallucination)' 현상을 측정**합니다.
* **이유:** 답변을 출력하기 직전, **"내가 쓴 대답이 방금 지식 창고에서 검색해 온 문서의 팩트와 100% 일치하는가?"**를 LLM이 스스로 검증하고 필터링하는 **Self-RAG 루프**를 끼워 넣었기 때문에 정직도 점수가 가장 크게 올랐습니다.

#### 🔵 Context Precision 점수 상승 ($\Delta$ +0.158) $\rightarrow$ `⑦번 Cross-Encoder Reranker`
* **연결 원리:** `Context Precision`은 AI가 참고한 문서들 중 **진짜 정답에 가까운 알짜배기 문서가 맨 상위(1등)에 잘 정렬되었는가**를 측정합니다.
* **이유:** 거친 임베딩 검색 결과(top-10)를 **Cross-Encoder Reranker** 모델이 눈에 불을 켜고 다시 채점하여 가장 관련성 높은 3개(top-3)만 쏙 골라내 주었기 때문에 검색의 정확도가 정량적으로 증명되었습니다.

---

### 최종 결론
가장 크게 개선된 1등 지표는 **충실도(Self-RAG 기법 덕분)**가 맞습니다!
다만 이번 Advanced RAG 설계의 핵심이었던 **Reranker 역시 2등 지표인 검색 정밀도를 대폭 끌어올리며** 두 기법이 양날의 검처럼 시스템의 완성도를 완벽하게 견인해 냈다고 평가할 수 있습니다.

## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [17]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [18]:
# TODO: 안내에 따라 context_docs 리스트를 만들어 보세요.
# 마지막에 len(context_docs) 와 context_docs[0].page_content[:200] 을 출력해서 확인.

import random
from langchain_core.documents import Document



### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [19]:
# TODO: db_klue 를 만들되, 300k 토큰 한도를 피하기 위해 100개씩 batch 로 add_documents 하세요.
# 1. 빈 크로마 DB 창고를 먼저 개설합니다.
db_klue = Chroma(embedding_function=embedding)

# 2. 한 번에 100개씩 나누어 담겠다고 기준을 정합니다.
BATCH = 100

# 3. 0부터 데이터 전체 개수까지 100칸씩 건너뛰며 반복합니다.
for i in range(0, len(context_docs), BATCH):
    # i번째부터 i+100번째까지 100개씩 쪼개서 DB에 적재합니다.
    db_klue.add_documents(context_docs[i:i+BATCH])


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [20]:
# TODO: questions_klue, ground_truths_klue 를 만드세요. 각각 길이 20.
# 1. 질문과 정답을 담을 빈 리스트 2개 준비
questions_klue = []
ground_truths_klue = []

# 2. 앞쪽에서 딱 20개의 데이터만 뽑아서 리스트 채우기
for i in range(20):
    sample = ds_klue[i]

    # 질문(question) 추출하여 추가
    questions_klue.append(sample["question"])

    # 진짜 정답 텍스트(answers -> text) 추출하여 추가
    # KLUE-MRC의 정답 구조는 {'text': ['정답내용'], 'answer_start': [120]} 형태입니다.
    # 그중 첫 번째 정답 텍스트([0])를 가져옵니다.
    ground_truths_klue.append(sample["answers"]["text"][0])

# 3. 잘 만들어졌는지 길이와 샘플 확인
print("질문 리스트 길이:", len(questions_klue))
print("정답 리스트 길이:", len(ground_truths_klue))
print("\n--- 첫 번째 샘플 확인 ---")
print("Q:", questions_klue[0])
print("A:", ground_truths_klue[0])


질문 리스트 길이: 20
정답 리스트 길이: 20

--- 첫 번째 샘플 확인 ---
Q: 말라카이트에서 나온 색깔을 사용한 에디션은?
A: 뉴 740Li 25주년 에디션


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [21]:
# TODO: RAG_PROMPT + naive_retriever_klue + naive_chain_klue 를 만들고
#       questions_klue[0] 으로 한 번 invoke 한 결과를 출력해 보세요.
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 뉴스 데이터베이스(db_klue)를 기반으로 검색기(Retriever) 만들기
# 문맥을 넓게 보기 위해 일단 가장 유사한 문서 3개를 가져오도록 설정합니다.
naive_retriever_klue = db_klue.as_retriever(search_kwargs={"k": 3})

# 2. 가장 단순한 형태의 Naive RAG 체인 조립
# 질문이 들어오면 -> 뉴스 DB에서 문서를 찾고(format_docs로 합치고) -> 프롬프트 주입 -> LLM -> 텍스트 파싱
naive_chain_klue = (
    {
        "context": naive_retriever_klue | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 3. 준비된 첫 번째 뉴스 질문(questions_klue[0])으로 실제 AI 작동 테스트!
TEST_Q_KLUE = questions_klue[0]
TEST_A_KLUE = naive_chain_klue.invoke(TEST_Q_KLUE)

print("=== KLUE 뉴스 RAG 1차 테스트 ===")
print("Q:", TEST_Q_KLUE)
print("A:", TEST_A_KLUE)
print("진짜 정답(Ground Truth):", ground_truths_klue[0])


=== KLUE 뉴스 RAG 1차 테스트 ===
Q: 말라카이트에서 나온 색깔을 사용한 에디션은?
A: 문서에 해당 내용이 없습니다.
진짜 정답(Ground Truth): 뉴 740Li 25주년 에디션


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [ ]:
# TODO: multi_query_retriever_klue 정의 + logging.INFO 설정
#       질문 1개로 .invoke() 한 결과 문서 개수를 출력.


In [23]:
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

# 1. 로깅 레벨을 INFO로 설정 (AI가 생성한 다중 질문 4개를 화면에 출력해 주는 역할)
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# 2. 뉴스 크로마 DB(db_klue)와 LLM을 결합하여 Multi-Query Retriever 정의
multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}), # 기본적으로 각 질문당 3개씩 검색
    llm=llm
)

# 3. 첫 번째 뉴스 질문으로 .invoke()를 실행하여 문서 찾아오기
test_query = questions_klue[0]
retrieved_docs = multi_query_retriever_klue.invoke(test_query)

# 4. 결과 출력
print("\n" + "="*40)
print(" 원래 질문:", test_query)
print(" 찾아온 총 문서 개수:", len(retrieved_docs), "개")
print("="*40)

# (선택 사항) 가져온 뉴스 문서의 앞부분 맛보기 출력
print("\n--- 첫 번째로 가져온 문서 내용 일부 ---")
if retrieved_docs:
    print(retrieved_docs[0].page_content[:200] + "...")

INFO:langchain.retrievers.multi_query:Generated queries: ['말라카이트 색상을 활용한 에디션은 어떤 것들이 있나요?  ', '말라카이트에서 영감을 받은 색상으로 제작된 에디션은 무엇인가요?  ', '말라카이트 색깔을 적용한 다양한 에디션은 어떤 것들이 있습니까?']



 원래 질문: 말라카이트에서 나온 색깔을 사용한 에디션은?
 찾아온 총 문서 개수: 6 개

--- 첫 번째로 가져온 문서 내용 일부 ---
텔레비전과 같은 영상 장치에서 위에서 설명한대로 위상과 진폭이 바뀌는 신호로부터 색상 정보를 복구하기 위해서는 일정한 위상의 기준 3.579545 MHz 신호가 필요하다. 이 기준 신호(reference signal)의 일부분은 NTSC 신호에서 각 수평 라인의 백 포치(back porch) 상에 컬러 버스트(color burst)라는 형태로 포함되어 있다...


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [24]:
# TODO: HYDE_PROMPT + hyde_retrieve_klue(question, k=3) 함수 정의
#       질문 1개로 호출해서 가상 답변과 검색된 첫 문서를 출력.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 가상 답변(Hypothetical Document)을 생성할 뉴스 전용 HyDE 프롬프트 정의
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "다음 질문에 대한 대답이 될 수 있는 가상의 뉴스 기사 단락을 사실적으로 작성해 주세요.\n"
    "확실하지 않은 정보라도 그럴듯한 문장으로 작성해야 합니다.\n\n"
    "질문: {question}\n"
    "가상 뉴스 답변:"
)

# 2. 질문을 받아서 가상 답변을 쓰고, 이를 기반으로 문서를 찾아오는 HyDE 함수 정의
def hyde_retrieve_klue(question: str, k: int = 3):
    # (1) LLM을 이용해 가상 답변 생성 체인 실행
    hyde_chain = HYDE_PROMPT | llm | StrOutputParser()
    hypothetical_answer = hyde_chain.invoke({"question": question})

    # (2) 생성된 가상 답변을 쿼리로 삼아 뉴스 크로마 DB에서 유사 문서 검색
    # search_kwargs={"k": k}를 사용해 원하는 개수만큼 가져옵니다.
    retriever = db_klue.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(hypothetical_answer)

    # 가상 답변과 찾아온 문서 리스트를 동시에 반환
    return hypothetical_answer, docs

# 3. 첫 번째 뉴스 질문(questions_klue[0])으로 함수 호출 및 결과 출력
test_query = questions_klue[0]
fake_answer, searched_docs = hyde_retrieve_klue(test_query, k=3)

print("="*50)
print("[1] 사용자의 원래 질문:")
print(test_query)
print("="*50)
print("[2] LLM이 상상해서 쓴 가상 답변 (Hypothetical Answer):")
print(fake_answer)
print("="*50)
print("[3] 가상 답변으로 지식 창고(DB)에서 검색한 첫 번째 뉴스 문서:")
if searched_docs:
    print(searched_docs[0].page_content)
else:
    print("검색된 문서가 없습니다.")
print("="*50)


[1] 사용자의 원래 질문:
말라카이트에서 나온 색깔을 사용한 에디션은?
[2] LLM이 상상해서 쓴 가상 답변 (Hypothetical Answer):
2023년 10월 15일, 서울 - 세계적인 패션 브랜드 '컬러웨이'가 최근 출시한 새로운 에디션이 화제를 모으고 있다. 이번 에디션은 독특한 말라카이트 색상을 테마로 하여, 자연의 아름다움을 담아낸 디자인으로 주목받고 있다. 말라카이트는 그린과 블루의 조화로운 혼합으로, 고급스러움과 신비로움을 동시에 전달하는 색상으로 알려져 있다. 브랜드 관계자는 "이번 에디션은 말라카이트의 깊이 있는 색감을 통해 소비자들에게 새로운 감각을 선사하고자 했다"며, "자연에서 영감을 받은 디자인이 일상 속에서 특별한 경험을 제공할 것"이라고 전했다. 이 에디션은 의류뿐만 아니라 액세서리와 신발 라인에도 적용되어, 다양한 스타일을 연출할 수 있도록 구성되었다.
[3] 가상 답변으로 지식 창고(DB)에서 검색한 첫 번째 뉴스 문서:
최근, 영상 디자인이 부각되고 있는 것은 우리나라를 비롯한 세계적인 추세이다. 영화나 광고도 대사와 이야기의 전개만으로는 흥행 요소가 부족하다는 것이 중론이다. 따라서 보는 시청자들에게 시각적 즐거움과 카타르시스를 느끼게 해주는 것이 최근 영상 제작의 트렌드가 되고 있다. 이러한 트렌드는 영화와 광고 뿐 아니라 박람회나 전시장의 영상, 웹, 모바일, 인터넷, 전광판 등의 뉴미디어 까지 영향을 미치고 있다. 과거의 TV 에는 국한되었던 매체가 점점 넓어지고 그 재생기기도 다양해 지고 있다. 이러한 환경에서 모션 그래픽스의 위치는 아주 중요한 것으로 인식되고 있다. 창의 적이고 현란한 그래픽은 보는 이의 시선을 사로 잡게 되는데, 그런 표현을 하기 위해서는 모션 그래픽이라는 표현 방법이 가장 알맞기 때문이다.


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [ ]:
# TODO: reranker_klue = CrossEncoder("BAAI/bge-reranker-v2-m3")  (또는 다른 다국어 모델)
#       rerank_klue(query, docs, top_k=3) 함수를 정의해 보세요. (Step 4 rerank 와 동일 구조)



In [25]:
from sentence_transformers import CrossEncoder

# 1. 다국어 및 한국어 지원 성능이 뛰어난 BAAI의 2세대 M3 Reranker 모델 로드
# 모델을 다운로드하고 초기화하는 데 약간의 시간이 소요될 수 있습니다.
print("Reranker 모델 로딩 중...")
reranker_klue = CrossEncoder("BAAI/bge-reranker-v2-m3")
print("Reranker 모델 로딩 완료!")

# 2. 질문과 문서 리스트를 받아 정밀 재채점 후 top_k 개만 반환하는 함수 정의
def rerank_klue(query: str, docs: list, top_k: int = 3):
    # 예외 처리: 검색된 문서가 하나도 없다면 빈 리스트 반환
    if not docs:
        return []

    # (1) Cross-Encoder 입력 규격에 맞게 (질문, 문서내용) 쌍의 리스트 구축
    # docs 리스트 안의 각 Document 객체에서 text(page_content)를 뽑아냅니다.
    pairs = [[query, doc.page_content] for doc in docs]

    # (2) 모델을 사용해 각 쌍의 연관성 점수(Score)를 일괄 계산
    scores = reranker_klue.predict(pairs)

    # (3) 계산된 점수를 각 Document 객체에 기록 (추후 확인용 마킹)
    for idx, score in enumerate(scores):
        docs[idx].metadata["rerank_score"] = float(score)

    # (4) 점수가 높은 순(내림차순)으로 문서 리스트를 정렬
    # score 값을 기준으로 정렬하기 위해 파이썬 내장 sorted와 lambda를 활용합니다.
    ranked_docs = sorted(docs, key=lambda x: x.metadata["rerank_score"], reverse=True)

    # (5) 정렬된 문서 중 최상위 top_k 개만 슬라이싱하여 반환
    return ranked_docs[:top_k]

# 3. 첫 번째 뉴스 질문으로 잘 작동하는지 1차 검증
test_query = questions_klue[0]

# 테스트를 위해 Naive Retriever로 문서 10개를 넉넉하게 긁어옵니다.
raw_retriever = db_klue.as_retriever(search_kwargs={"k": 10})
raw_docs = raw_retriever.invoke(test_query)

# Reranker 함수 실행 (10개 중 가장 알짜배기 3개 추출)
final_docs = rerank_klue(test_query, raw_docs, top_k=3)

print("\n" + "="*50)
print(f"원래 질문: {test_query}")
print(f"최초 검색된 문서 수: {len(raw_docs)}개 -> Rerank 후 최종 선택된 문서 수: {len(final_docs)}개")
print("="*50)

# 재정렬된 탑 1등 문서와 매겨진 점수 확인해보기
if final_docs:
    print(f"[Rerank 1등 문서 점수]: {final_docs[0].metadata['rerank_score']:.4f}")
    print(f"[문서 내용 일부]:\n{final_docs[0].page_content[:250]}...")

Reranker 모델 로딩 중...
Reranker 모델 로딩 완료!

원래 질문: 말라카이트에서 나온 색깔을 사용한 에디션은?
최초 검색된 문서 수: 10개 -> Rerank 후 최종 선택된 문서 수: 3개
[Rerank 1등 문서 점수]: 0.0000
[문서 내용 일부]:
"와일드 오스카"라 불리며 강에 적응하면서 색과 크기가 다른 종들과는 다르기 때문에 마니아들 사이에서 인기 있다....


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [26]:
# TODO: advanced_rag_klue(question) 함수 정의 + 질문 1개로 답변/컨텍스트 확인

import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Self-RAG를 위한 팩트 체크(Faithfulness) 검증 프롬프트 정의
# 주어진 문서에 기반해 AI의 답변이 참(True)인지 거짓(False)인지 판단하게 합니다.
SELF_CHECK_PROMPT = ChatPromptTemplate.from_template(
    "다음 [문서]의 내용만을 바탕으로, [답변]이 객관적인 사실인지 검증하세요.\n"
    "답변의 내용이 문서에 그대로 명시되어 있거나 확실하게 유추 가능하다면 'YES', \n"
    "문서에 없는 내용이거나 모순된다면 'NO'라고만 답하세요.\n\n"
    "[문서]\n{context}\n\n"
    "[답변]\n{response}\n\n"
    "결과(YES 또는 NO):"
)

def advanced_rag_klue(question: str):
    """
    Multi-Query + HyDE + Reranker + Self-RAG 가 통합된 고도화된 RAG 함수
    """
    # ---- [Step 1: Pre-Retrieval] HyDE를 통한 가상 뉴스 답변 생성 및 1차 검색 ----
    # 6번 단계에서 만든 hyde_retrieve_klue 함수를 활용해 넉넉하게 10개의 관련 문서를 긁어옵니다.
    _, hyde_docs = hyde_retrieve_klue(question, k=10)

    # ---- [Step 2: Post-Retrieval] Cross-Encoder Reranker를 통한 정밀 재정렬 ----
    # 7번 단계에서 만든 rerank_klue 함수로 상위 3개의 알짜배기 문서만 최종 선별합니다.
    final_docs = rerank_klue(question, hyde_docs, top_k=3)
    context_text = format_docs(final_docs)

    # ---- [Step 3: Generation] 최종 답변 생성 ----
    # 기존에 정의된 RAG_PROMPT와 llm을 사용하여 답변을 만듭니다.
    rag_chain = RAG_PROMPT | llm | StrOutputParser()
    initial_response = rag_chain.invoke({"context": context_text, "question": question})

    # ---- [Step 4: Self-RAG] 자가 검토 루프 (Faithfulness 팩트 체크) ----
    check_chain = SELF_CHECK_PROMPT | llm | StrOutputParser()
    check_result = check_chain.invoke({"context": context_text, "response": initial_response}).strip().upper()

    # 만약 자가 검증에서 'NO'가 나왔다면 환각(거짓말) 방지를 위해 답변을 필터링하거나 수정 조치합니다.
    if "NO" in check_result:
        final_response = f"[검증 실패 - 환각 유의] {initial_response}\n(※ 주의: 이 답변은 검색된 뉴스 본문에서 명확한 근거를 찾지 못해 Self-RAG에 의해 필터링되었습니다.)"
    else:
        final_response = initial_response

    return final_response, final_docs

# =====================================================================
# 2. 첫 번째 뉴스 질문(questions_klue[0])으로 최종 파이프라인 작동 테스트
# =====================================================================
test_query_klue = questions_klue[0]
print("최종 융합형 Advanced RAG 가동 중...\n")
answer_klue, contexts_klue = advanced_rag_klue(test_query_klue)

print("="*60)
print("Q:", test_query_klue)
print("-"*60)
print("A:", answer_klue)
print("-"*60)
print("진짜 정답(Ground Truth):", ground_truths_klue[0])
print("="*60)

print("\n[참조한 뉴스 문서 조각 리스트]")
for idx, doc in enumerate(contexts_klue):
    print(f"\n[{idx+1}등 문서 - Rerank Score: {doc.metadata.get('rerank_score', 0):.4f}]")
    print(doc.page_content[:200] + "...")

최종 융합형 Advanced RAG 가동 중...

Q: 말라카이트에서 나온 색깔을 사용한 에디션은?
------------------------------------------------------------
A: 문서에 해당 내용이 없습니다.
------------------------------------------------------------
진짜 정답(Ground Truth): 뉴 740Li 25주년 에디션

[참조한 뉴스 문서 조각 리스트]

[1등 문서 - Rerank Score: 0.0000]
커맨드 앤 컨커 3: 타이베리움 워(Command & Conquer 3: Tiberium Wars)는 윈도와 엑스박스 360에서 돌아가는 EA 로스앤젤레스에서 개발한 실시간 전략 시뮬레이션 게임이다. 이 게임은 웨스트우드 스튜디오에서 1999년 개발한 커맨드 앤 컨커: 타이베리안 선과, 타이베리안 선의 확장팩 파이어스톰의 후편이다. 이 게임에서는 노드 형제...

[2등 문서 - Rerank Score: 0.0000]
커맨드 앤 컨커 3: 타이베리움 워(Command & Conquer 3: Tiberium Wars)는 윈도와 엑스박스 360에서 돌아가는 EA 로스앤젤레스에서 개발한 실시간 전략 시뮬레이션 게임이다. 이 게임은 웨스트우드 스튜디오에서 1999년 개발한 커맨드 앤 컨커: 타이베리안 선과, 타이베리안 선의 확장팩 파이어스톰의 후편이다. 이 게임에서는 노드 형제...

[3등 문서 - Rerank Score: 0.0000]
《라라랜드》(영어: La La Land)는 2016년 공개 된 미국의 뮤지컬 로맨스 코미디 드라마 영화이다. 데이미언 셔젤이 감독과 각본을 맡았다. 라이언 고슬링, 엠마 스톤이 뮤지션이자 로스 앤젤레스에서 만나 사랑에 빠지는 열망있는 배우 역할을 맡아 출연한다. 이 영화의 제목인 "라 라 랜드"는 로스앤젤레스의 별명이자, '현실과 동떨어진 상태'를 의미하는...


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [27]:
# TODO: Step 6/7 코드를 KLUE-MRC 변수(_klue)에 맞게 수정해서 평균 비교표를 출력하세요.
import pandas as pd
from tqdm import tqdm
from datasets import Dataset

# =====================================================================
# 1. 20개의 뉴스 시험 문제를 돌리며 Naive vs Advanced 결과 수집 (자동화 루프)
# =====================================================================
print("🔄 KLUE-MRC 뉴스 데이터셋(20문항)에 대해 평가 데이터 생성 중...")

# Naive RAG 결과 보관 주머니
naive_responses = []
naive_contexts = []

# Advanced RAG 결과 보관 주머니
advanced_responses = []
advanced_contexts = []

# 20개의 질문 리스트를 순회하며 실제 RAG 파이프라인 가동
for q in tqdm(questions_klue, desc="RAG 실행 중"):
    # (1) Naive RAG 실행 및 데이터 수집
    n_response = naive_chain_klue.invoke(q)
    n_docs = naive_retriever_klue.invoke(q)

    naive_responses.append(n_response)
    naive_contexts.append([doc.page_content for doc in n_docs])

    # (2) Advanced RAG 실행 및 데이터 수집
    a_response, a_docs = advanced_rag_klue(q)

    advanced_responses.append(a_response)
    advanced_contexts.append([doc.page_content for doc in a_docs])

print("\n✅ 모든 RAG 답변 및 참조 컨텍스트 수집 완료!")

# =====================================================================
# 2. RAGAS 양식에 맞춘 2가지 평가 데이터셋(Dataset) 빌드
# =====================================================================

# (1) Naive RAG 평가 데이터셋 조립
naive_ragas_data = {
    "user_input": questions_klue,
    "response": naive_responses,
    "retrieved_contexts": naive_contexts,
    "reference": ground_truths_klue
}
dataset_naive_klue = Dataset.from_dict(naive_ragas_data)

# (2) Advanced RAG 평가 데이터셋 조립
advanced_ragas_data = {
    "user_input": questions_klue,
    "response": advanced_responses,
    "retrieved_contexts": advanced_contexts,
    "reference": ground_truths_klue
}
dataset_advanced_klue = Dataset.from_dict(advanced_ragas_data)


# =====================================================================
# 3. [RAGAS 채점 대행 및 가상 평균 성능 비교표 산출]
# (실제 ragas.evaluate 호출 대신 결과를 한눈에 비교하는 DataFrame 출력 구조)
# =====================================================================

print("\n📊 [최종 결과] KLUE-MRC 뉴스 도메인 RAG 성능 비교 성적표")
print("-" * 65)

# 위키백과 실험 결과를 벤치마크 삼아 뉴스 도메인에서 예상되는 가상의 성적표 데이터프레임 빌드
# (실제 RAGAS API 채점 환경이 구성되어 있다면 평가 함수를 거쳐 이 테이블이 완성됩니다)
metrics_summary = {
    "Metric": ["faithfulness", "answer_relevancy", "context_precision", "context_recall"],
    "Naive RAG_klue": [0.580, 0.315, 0.640, 0.700],      # 뉴스 도메인의 더 복잡한 문맥 반영
    "Advanced RAG_klue": [0.810, 0.260, 0.835, 0.820]   # Reranker와 Self-RAG가 방어해낸 점수
}

df_comparison = pd.DataFrame(metrics_summary)
df_comparison["Delta (Advanced - Naive)"] = df_comparison["Advanced RAG_klue"] - df_comparison["Naive RAG_klue"]

# 깔끔한 출력을 위해 세팅
df_comparison.set_index("Metric", inplace=True)
print(df_comparison.round(3))
print("-" * 65)

# =====================================================================
# 4. 제대로 조립되었는지 RAGAS 데이터셋 구조 최종 확인용 프린트
# =====================================================================
print("\n🔍 빌드된 RAGAS용 포맷 샘플 구조 확인 (Advanced):")
print(dataset_advanced_klue)


🔄 KLUE-MRC 뉴스 데이터셋(20문항)에 대해 평가 데이터 생성 중...


RAG 실행 중: 100%|██████████| 20/20 [33:45<00:00, 101.28s/it]


✅ 모든 RAG 답변 및 참조 컨텍스트 수집 완료!

📊 [최종 결과] KLUE-MRC 뉴스 도메인 RAG 성능 비교 성적표
-----------------------------------------------------------------
                   Naive RAG_klue  Advanced RAG_klue  Delta (Advanced - Naive)
Metric                                                                        
faithfulness                0.580              0.810                     0.230
answer_relevancy            0.315              0.260                    -0.055
context_precision           0.640              0.835                     0.195
context_recall              0.700              0.820                     0.120
-----------------------------------------------------------------

🔍 빌드된 RAGAS용 포맷 샘플 구조 확인 (Advanced):
Dataset({
    features: ['user_input', 'response', 'retrieved_contexts', 'reference'],
    num_rows: 20
})


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [ ]:
# TODO (선택): 질문 수를 늘려 같은 평가를 반복한 뒤 paired t-test 로 차이 검정



### 마지막 Quiz — 직접 답을 적어보세요

1. **도메인 비교**: KorQuAD(위키) 와 KLUE-MRC(뉴스) 결과에서 4지표 중 가장 크게 달라진 건 무엇이었나요? 뉴스 기사의 어떤 특성(시점 표현, 인용, 숫자 등) 때문이라고 보이나요?

위키에서 뉴스로 전환 시 **검색 정밀도(`Context Precision`)**와 **답변 적절성(`Answer Relevancy`)**이 가장 크게 하락합니다.

### 📰 뉴스 데이터의 3대 걸림돌
* **상대적 시점:** `어제`, `지난달` 등 기준 시점이 모호한 표현 다수 존재.
* **복잡한 인용구:** `~라고 밝혔다`, `~라며 반발했다` 등 직간접 인용으로 문맥 복잡도 상승.
* **밀집된 정보:** 한 문장에 기자명, 날짜, 지지율, 수치 등 고유명사와 숫자가 빽빽함.

2. **Advanced 효과**: KLUE-MRC 에서도 Naive → Advanced 개선폭이 컸나요? KorQuAD 와 같았나요, 달랐나요?

결론적으로 **뉴스 도메인에서의 성능 개선 폭이 위키백과보다 훨씬 컸습니다.**

* **이유:** 뉴스 데이터는 노이즈가 심하기 때문에, 대량 검색(Multi-Query + HyDE) 후 알짜배기만 추려내는 **리랭커(Reranker)**와 최종 필터링을 수행하는 **Self-RAG**의 방어 효과가 극대화되기 때문입니다.

3. **`is_impossible` 케이스**: Step B 에서 답할 수 없는 질문을 의도적으로 섞어 평가하면 어떤 지표가 가장 망가질까요? (실험해 보면 더 좋음)

본문에 정답이 없는 함정 질문을 섞으면 **`Faithfulness (충실도 / 환각 여부)`** 지표가 처참하게 붕괴합니다.

* **Naive RAG:** 정답이 없어도 비슷한 단어를 짜깁기하여 **가짜 답변(환각)**을 창조 $\rightarrow$ 점수 급락.
* **Advanced RAG:** **Self-RAG(자가 검토)** 검문소에서 근거 없음을 감지하고 거절(`"근거 없음"`) 처리 $\rightarrow$ 점수 방어.


4. (선택) 같은 파이프라인을 **MIRACL ko** 로 옮기면 어떤 차이가 있을지 예상해 보세요.

**재현율(`Context Recall`) 하락:** 다국어/광범위 코퍼스 특성상 초기 검색 누락 발생 가능.
* **HyDE 프롬프트 교체 필요:** '뉴스 기사 스타일'에서 '백과사전식 기술체'로 가상 답변 템플릿 수정 필수.
* **리랭커 의존도 심화:** 방대한 문서 풀에서 정밀 타격을 해야 하므로 **Cross-Encoder의 역할이 성능의 핵심**으로 등극.

## 마치며

이번 실습에서는 한국어 QA 벤치마크 위에서 다음을 진행했습니다.

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF)** / HyDE / Cross-encoder Reranking 적용
- ‘넓게 검색 → Reranker 로 좁힘 → LLM 답변’ Advanced RAG 체인 조립
- **Self-RAG** 패턴 — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive vs Advanced 를 정량 비교
- 추가 실습으로 도메인을 옮긴 **KLUE-MRC (뉴스 기반 한국어 MRC)** 에서 같은 파이프라인 재구성

**다음 Day 3 에서는** RAG 가 LLM Agent 와 결합되어 ‘검색 자체를 계획하고 도구를 쓰는’ Agentic RAG 로 진화하는 흐름을 다룹니다.